In [1]:
# Install the core YOLO framework
!pip install ultralytics -q

import os
from ultralytics import YOLO

# Verify the GPU execution route is active
import torch
print(f"✅ Setup complete. Using GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Active (Turn on accelerator!)'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Setup complete. Using GPU: Tesla T4


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="M2hZer8PzPBpFRWbCD3q")
project = rf.workspace("madhusudans-workspace-31zqm").project("cc_ai_model_v6")
version = project.version(1)
dataset = version.download("yolov8")
                

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 111.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CC_AI_Model_V6-1 in yolov8:: 100%|██████████| 75797/75797 [00:08<00:00, 9316.10it/s] 


In [3]:
import yaml

# Locate the configuration file dynamically
yaml_path = os.path.join(dataset.location, "data.yaml")

with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Hardcode absolute paths directly to Kaggle's working directory
config['train'] = os.path.join(dataset.location, "train", "images")
config['val']   = os.path.join(dataset.location, "valid", "images")
config['test']  = os.path.join(dataset.location, "test", "images")

with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

print(f"✏️ data.yaml updated with absolute Kaggle environment paths!")

✏️ data.yaml updated with absolute Kaggle environment paths!


In [4]:
import os
from ultralytics import YOLO

def train_custom_yolo():
    # --- 1. Find your data.yaml automatically ---
    # Change this to your folder name if your file is inside a specific directory (e.g., "dataset")
    target_file = "data.yaml" 
    yaml_path = None

    # Check current directory first
    if os.path.exists(target_file):
        yaml_path = os.path.abspath(target_file)
    else:
        # Walk through directories to find it for you
        for root, dirs, files in os.walk("."):
            if target_file in files:
                yaml_path = os.path.abspath(os.path.join(root, target_file))
                break

    if not yaml_path:
        raise FileNotFoundError(
            f"❌ Could not find '{target_file}' anywhere in your workspace. "
            f"Please ensure it is uploaded or check the spelling. "
            f"Current directory contents: {os.listdir('.')}"
        )
    
    print(f"✅ Found configuration file at: {yaml_path}")

    # --- 2. Load and Train the Model ---
    model = YOLO("yolov8n.pt")

    results = model.train(
        data=yaml_path,             # Uses the verified path found above
        epochs=100,                 # 100 epochs optimization timeline
        imgsz=640,                  # Keeps target features sharp
        batch=32,                   # Safe baseline for standard VRAM
        device=[0, 1],                   # Uses GPU 0
        
        # --- Advanced Hyperparameters for 80%+ mAP ---
        cos_lr=True,                # Smooth cosine decay over the 100 epochs
        lr0=0.01,                   
        lrf=0.01,                   
        warmup_epochs=3,            # Gives the main training loop maximum room
        
        # --- Class & Box Loss Tuning ---
        box=8.5,                    # Forced tighter bounding boxes on objects
        cls=1.5,                    # Higher penalty for mixing up classes
        
        # --- Aggressive Augmentations for Edge Cases (Balaclava) ---
        mosaic=1.0,                 
        mixup=0.15,                 
        scale=0.5,                  
        
        # --- The Final Stretch ---
        close_mosaic=15,            # Turns off mosaic at epoch 85 to lock in box precision
        deterministic=True,         
        project="YOLOv8_Security",  
        name="v8n_100e_optimized"    
    )

if __name__ == "__main__":
    train_custom_yolo()



✅ Found configuration file at: /kaggle/working/CC_AI_Model_V6-1/data.yaml
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=8.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=1.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/CC_AI_Model_V6-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mix

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:99: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /root/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at /pytorch/aten/src/ATen/native/cuda/jit_utils.cpp:1487.)
  inter = (torch.min(a2, b2) - torch.max(a1, b1)).clamp_(0).prod(2)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 3.6it/s 10.5s
                   all       2379       4732      0.691      0.624      0.678      0.406

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      2.63G      1.631      4.996      1.478         32        640: 100% ━━━━━━━━━━━━ 1036/1036 2.7it/s 6:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 3.9it/s 9.7s
                   all       2379       4732       0.68      0.589      0.657      0.376

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      2.63G       1.74      5.159      1.562         31        640: 100% ━━━━━━━━━━━━ 1036/1036 2.9it/s 5:57
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 3.9it/s 9.7s
                   all       2379       47